# Chapter 2 — Data Manipulation: Network Traffic Data Cleaning

## 📋 Latihan Soal

Bersihkan dan manipulasi data network traffic sebelum analisis lebih lanjut. Data mentah biasanya berisi missing values, duplikat, dan format yang tidak konsisten.

### Dataset
- `sensor_a`: 500 records (primary monitoring, ada missing values & NaN)
- `sensor_b`: 200 records (batch monitoring baru)
- `sensor_c`: 300 records (duplikat yang sudah disiapkan dari sensor A & B)
- `threat_intel`: 50 records port mapping (port → risk level, attack type)

In [ ]:
# ============================================
# SETUP: Generate Dataset untuk Data Manipulation
# ============================================
import numpy as np
import pandas as pd

np.random.seed(42)

# === Sensor A: Internal Network ===
n = 2000
sensor_a = pd.DataFrame({
    'timestamp': pd.date_range('2024-01-15 08:00', periods=n, freq='30s'),
    'src_ip': [f'192.168.1.{np.random.randint(1,255)}' for _ in range(n)],
    'dst_port': np.random.choice([80, 443, 22, 3389, 8080, 53], n),
    'packet_rate': np.random.exponential(100, n),
    'bytes_sent': np.random.exponential(50000, n),
    'protocol': np.random.choice(['TCP', 'UDP', 'ICMP'], n, p=[0.7, 0.2, 0.1]),
    'flag': np.random.choice(['SYN', 'ACK', 'FIN', 'RST'], n, p=[0.4, 0.3, 0.2, 0.1]),
    'sensor_id': 'SENSOR_A'
})

# === Sensor B: DMZ ===
n2 = 1500
sensor_b = pd.DataFrame({
    'timestamp': pd.date_range('2024-01-15 08:00', periods=n2, freq='45s'),
    'src_ip': [f'10.0.{np.random.randint(0,5)}.{np.random.randint(1,255)}' for _ in range(n2)],
    'dst_port': np.random.choice([80, 443, 25, 587, 993], n2),
    'packet_rate': np.random.exponential(200, n2),
    'bytes_sent': np.random.exponential(80000, n2),
    'protocol': np.random.choice(['TCP', 'UDP', 'ICMP'], n2, p=[0.8, 0.15, 0.05]),
    'flag': np.random.choice(['SYN', 'ACK', 'FIN', 'RST'], n2, p=[0.5, 0.25, 0.15, 0.1]),
    'sensor_id': 'SENSOR_B'
})

# === Sensor C: External Gateway ===
n3 = 1800
sensor_c = pd.DataFrame({
    'timestamp': pd.date_range('2024-01-15 08:00', periods=n3, freq='35s'),
    'src_ip': [f'{np.random.randint(1,223)}.{np.random.randint(0,256)}.{np.random.randint(0,256)}.{np.random.randint(1,255)}' for _ in range(n3)],
    'dst_port': np.random.choice([80, 443, 22, 3306, 5432, 6379], n3),
    'packet_rate': np.random.exponential(300, n3),
    'bytes_sent': np.random.exponential(120000, n3),
    'protocol': np.random.choice(['TCP', 'UDP', 'ICMP'], n3, p=[0.75, 0.2, 0.05]),
    'flag': np.random.choice(['SYN', 'ACK', 'FIN', 'RST'], n3, p=[0.45, 0.2, 0.15, 0.2]),
    'sensor_id': 'SENSOR_C'
})

# Add missing values (sensor glitches)
for df in [sensor_a, sensor_b, sensor_c]:
    for col in ['packet_rate', 'bytes_sent']:
        mask = np.random.choice(len(df), size=int(len(df)*0.05), replace=False)
        df.loc[mask, col] = np.nan

# Add duplicate rows (packet replay)
dup_a = sensor_a.sample(n=100, random_state=99)
dup_b = sensor_b.sample(n=80, random_state=99)

# Threat intelligence data (untuk merging)
threat_intel = pd.DataFrame({
    'dst_port': [22, 3389, 3306, 5432, 6379, 8080, 25, 587],
    'port_name': ['SSH', 'RDP', 'MySQL', 'PostgreSQL', 'Redis', 'HTTP-Alt', 'SMTP', 'SMTP-Submit'],
    'risk_level': ['High', 'High', 'High', 'High', 'Critical', 'Medium', 'Medium', 'Low'],
    'common_attack': ['Brute Force', 'BlueKeep', 'SQL Injection', 'SQL Injection',
                      'Unauthorized Access', 'Web Exploit', 'Spam Relay', 'Phishing']
})

print("Dataset siap!")
print(f"Sensor A: {len(sensor_a)} rows")
print(f"Sensor B: {len(sensor_b)} rows")
print(f"Sensor C: {len(sensor_c)} rows")
print(f"Duplikat A: {len(dup_a)} rows")
print(f"Duplikat B: {len(dup_b)} rows")
print(f"Threat Intel: {len(threat_intel)} rows")

---
## Soal 1: Grouping — Agregasi per Time Window

Kelompokkan data `sensor_a` berdasarkan 5-menit time window. Hitung total packet_rate, rata-rata bytes_sent, dan jumlah unique src_ip per window.

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: df.resample('5min', on='timestamp').agg(...)

# Jawaban:
df_a_clean = sensor_a.dropna(subset=['packet_rate', 'bytes_sent'])
grouped = df_a_clean.resample('5min', on='timestamp').agg(
    total_packets=('packet_rate', 'sum'),
    avg_bytes=('bytes_sent', 'mean'),
    unique_ips=('src_ip', 'nunique'),
    connection_count=('packet_rate', 'count')
).dropna()

print(f"=== Agregasi 5-Menit Window ({len(grouped)} windows) ===")
print(grouped.head(10))
print(f"\nWindow dengan traffic tertinggi:")
peak = grouped['total_packets'].idxmax()
print(f"  Time: {peak}")
print(f"  Total Packets: {grouped.loc[peak, 'total_packets']:.0f}")
print(f"  Unique IPs: {grouped.loc[peak, 'unique_ips']}")

---
## Soal 2: Appending — Tambah Batch Data Baru

Ada batch monitoring baru (`sensor_b`). Append data sensor_b ke sensor_a untuk membuat consolidated dataset.

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: pd.concat([df1, df2], ignore_index=True)

# Jawaban:
consolidated = pd.concat([sensor_a, sensor_b], ignore_index=True)
print(f"Sensor A: {len(sensor_a)} rows")
print(f"Sensor B: {len(sensor_b)} rows")
print(f"Consolidated: {len(consolidated)} rows")
print(f"\nDistribusi per sensor:")
print(consolidated['sensor_id'].value_counts())

---
## Soal 3: Concatenating — Gabungkan 3 Sensor

Concatenate data dari ketiga sensor (A, B, C) menjadi satu DataFrame. Tambahkan duplikat yang sudah disiapkan, lalu cek total baris sebelum dan sesudah deduplication.

In [ ]:
# TULIS JAWABAN DI SINI

# Jawaban:
all_sensors = pd.concat([sensor_a, sensor_b, sensor_c, dup_a, dup_b], ignore_index=True)
print(f"Total sebelum deduplikasi: {len(all_sensors)} rows")

# Soal 7: Removing Duplicates (dikerjakan di sini)
before_dedup = len(all_sensors)
all_sensors = all_sensors.drop_duplicates()
after_dedup = len(all_sensors)

print(f"Total setelah deduplikasi: {after_dedup} rows")
print(f"Duplikat dihapus: {before_dedup - after_dedup} rows")
print(f"\nDistribusi per sensor setelah dedup:")
print(all_sensors['sensor_id'].value_counts())

---
## Soal 4: Merging — Join dengan Threat Intelligence

Merge data traffic dengan threat intelligence berdasarkan `dst_port`. Tambahkan informasi `port_name`, `risk_level`, dan `common_attack` ke setiap record.

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: pd.merge(df, threat_intel, on='dst_port', how='left')

# Jawaban:
merged = pd.merge(all_sensors, threat_intel, on='dst_port', how='left')

print(f"Shape setelah merge: {merged.shape}")
print(f"\n=== Sample data setelah merge ===")
print(merged[['sensor_id', 'dst_port', 'port_name', 'risk_level', 'common_attack']].head(10))

# Port yang tidak ada di threat intel
unmatched = merged['port_name'].isna().sum()
print(f"\n⚠️ Port yang tidak ada di threat intel: {unmatched} rows")

# Distribusi risk level
print(f"\nDistribusi Risk Level:")
print(merged['risk_level'].value_counts(dropna=False))

---
## Soal 5: Sorting — Urutkan Berdasarkan Waktu & Severity

Urutkan data berdasarkan `timestamp` (ascending), kemudian `risk_level` (Critical > High > Medium > Low) sebagai secondary sort.

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: pd.Categorical dengan ordered=True

# Jawaban:
risk_order = pd.CategoricalDtype(
    categories=['Critical', 'High', 'Medium', 'Low'],
    ordered=True
)
merged['risk_level'] = merged['risk_level'].astype(risk_order)

sorted_df = merged.sort_values(
    by=['timestamp', 'risk_level'],
    ascending=[True, True]
).reset_index(drop=True)

print("=== 15 Baris Pertama (urut timestamp, prioritas Critical→Low) ===")
cols = ['timestamp', 'sensor_id', 'dst_port', 'risk_level', 'common_attack', 'packet_rate']
print(sorted_df[cols].head(15))

---
## Soal 6: Categorising — Kategori Traffic Level

Kategorikan `packet_rate` ke dalam 4 level:
- **Low**: < 50
- **Medium**: 50 - 150
- **High**: 150 - 400
- **Critical**: > 400

Buat kolom baru `traffic_level`.

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: pd.cut() atau np.select()

# Jawaban:
df_for_categorize = sorted_df.dropna(subset=['packet_rate']).copy()

bins = [0, 50, 150, 400, float('inf')]
labels = ['Low', 'Medium', 'High', 'Critical']
df_for_categorize['traffic_level'] = pd.cut(
    df_for_categorize['packet_rate'],
    bins=bins,
    labels=labels,
    right=True
)

print("=== Distribusi Traffic Level ===")
print(df_for_categorize['traffic_level'].value_counts().sort_index())
print(f"\n=== Cross-tab: Traffic Level vs Risk Level ===")
crosstab = pd.crosstab(df_for_categorize['traffic_level'], df_for_categorize['risk_level'])
print(crosstab)

---
## Soal 8: Dropping Rows & Columns

1. Drop baris yang `packet_rate`-nya NaN (sensor failure)
2. Drop kolom `sensor_id` karena sudah tidak diperlukan setelah agregasi
3. Drop baris yang `risk_level`-nya NaN (port tidak dikenali) — opsional

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: df.dropna() dan df.drop(columns=...)

# Jawaban:
print(f"Shape sebelum drop: {sorted_df.shape}")
print(f"Rows dengan NaN packet_rate: {sorted_df['packet_rate'].isna().sum()}")

# Drop rows dengan NaN pada kolom kritis
dropped = sorted_df.dropna(subset=['packet_rate', 'bytes_sent'])
print(f"Shape setelah drop NaN: {dropped.shape}")

# Drop kolom yang tidak perlu
dropped = dropped.drop(columns=['sensor_id'])
print(f"Shape setelah drop kolom: {dropped.shape}")
print(f"Kolom tersisa: {list(dropped.columns)}")

---
## Soal 9: Changing Data Format

1. Konversi `timestamp` dari datetime ke string format 'YYYY-MM-DD HH:MM'
2. Konversi `packet_rate` dan `bytes_sent` ke integer
3. Buat kolom baru `hour` dan `minute` dari timestamp

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: dt.strftime(), dt.hour, astype()

# Jawaban:
df_formatted = dropped.copy()
df_formatted['timestamp_str'] = df_formatted['timestamp'].dt.strftime('%Y-%m-%d %H:%M')
df_formatted['packet_rate_int'] = df_formatted['packet_rate'].astype(int)
df_formatted['bytes_sent_int'] = df_formatted['bytes_sent'].astype(int)
df_formatted['hour'] = df_formatted['timestamp'].dt.hour
df_formatted['minute'] = df_formatted['timestamp'].dt.minute

print("=== Data Types Setelah Formatting ===")
print(df_formatted[['timestamp_str', 'packet_rate_int', 'bytes_sent_int', 'hour', 'minute']].dtypes)
print(f"\n=== Sample ===")
print(df_formatted[['timestamp_str', 'packet_rate_int', 'bytes_sent_int', 'hour', 'minute']].head())

print(f"\n=== Distribusi per Hour ===")
print(df_formatted['hour'].value_counts().sort_index())

---
## Soal 10: Replacing Data

Ada nilai `flag = 'RST'` yang seharusnya tidak muncul pada koneksi normal. Ganti flag 'RST' menjadi 'RESET_ANOMALY' untuk menandai anomali. Juga, ganti `risk_level` yang NaN menjadi 'Unknown'.

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: df.replace() atau df.loc[df['col'] == 'RST', 'col'] = 'RESET_ANOMALY'

# Jawaban:
df_replaced = df_formatted.copy()

# Ganti flag RST
rst_count = (df_replaced['flag'] == 'RST').sum()
df_replaced.loc[df_replaced['flag'] == 'RST', 'flag'] = 'RESET_ANOMALY'
print(f"Flag RST diganti: {rst_count} rows")

# Ganti NaN risk_level
na_risk = df_replaced['risk_level'].isna().sum()
df_replaced['risk_level'] = df_replaced['risk_level'].cat.add_categories(['Unknown']).fillna('Unknown')
print(f"Risk_level NaN diganti ke Unknown: {na_risk} rows")

print(f"\nFlag distribution:")
print(df_replaced['flag'].value_counts())

---
## Soal 11: Dealing with Missing Values

Handle missing values pada data asli (`sensor_a`) dengan 3 metode berbeda:
1. **Forward Fill** — Isi dengan nilai sebelumnya (cocok untuk time series)
2. **Mean Imputation** — Isi dengan rata-rata per sensor
3. **Interpolation** — Interpolasi linear berdasarkan waktu

Bandingkan hasilnya.

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: df.ffill(), df.fillna(df.mean()), df.interpolate()

# Jawaban:
sa = sensor_a.copy()
missing_count = sa['packet_rate'].isna().sum()
print(f"Missing values di packet_rate: {missing_count}")

# Metode 1: Forward Fill
ffill = sa['packet_rate'].ffill()
print(f"\n1. Forward Fill — NaN tersisa: {ffill.isna().sum()}")

# Metode 2: Mean Imputation
mean_fill = sa['packet_rate'].fillna(sa['packet_rate'].mean())
print(f"2. Mean Imputation — NaN tersisa: {mean_fill.isna().sum()}")
print(f"   Mean asli: {sa['packet_rate'].mean():.2f}")

# Metode 3: Interpolation
interp = sa['packet_rate'].interpolate(method='linear')
print(f"3. Interpolation — NaN tersisa: {interp.isna().sum()}")

# Bandingkan statistik
comparison = pd.DataFrame({
    'Original': sa['packet_rate'].describe(),
    'FFill': ffill.describe(),
    'Mean': mean_fill.describe(),
    'Interp': interp.describe()
}).round(2)
print(f"\n=== Perbandingan Statistik ===")
print(comparison)

---
## 📊 Ringkasan Chapter 2

Teknik data manipulation yang sudah dipelajari:
- **Grouping** → Agregasi traffic per 5-menit window
- **Appending/Concatenating** → Gabungkan data multi-sensor
- **Merging** → Enrichment dengan threat intelligence
- **Sorting** → Prioritaskan traffic berdasarkan severity
- **Categorising** → Klasifikasi traffic level dengan pd.cut()
- **Deduplication** → Hapus packet replay
- **Dropping** → Buang data corrupt dan kolom tidak relevan
- **Format conversion** → Time-based feature extraction
- **Replacing** → Tandai anomali pada flag dan handle unknown risk
- **Missing values** → 3 strategi imputasi untuk time series